In [ ]:
from pathlib import Path
ROOT = Path().resolve()
while not (ROOT / "DATA").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA = ROOT / "DATA"
print("CLEAN root:", ROOT)


In [1]:
import json
import pickle
import unicodedata
from collections import defaultdict
from pathlib import Path

import pandas as pd
from kg_gen.models import Graph

/home/ppoulenard/anaconda3/envs/LLM_BIAS_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============ CHOIX DE LA CATÉGORIE ============
CATEGORY = "danza"
MODELE = "mistral/mistral-small-2506"
MODELE_title = MODELE.split("/")[-1].replace("-", "_")

# ============ CHEMINS ============
BASE_DIR = ROOT
SUBSETS_DIR = DATA / "SUBSETS_with_relations" / "ARTICLES_SUBSETS_ES"
RESULTS_BASE = DATA / "SUBSETS_with_relations" / "GRAPHS"

CATEGORY_DIR = RESULTS_BASE / CATEGORY
OUTPUT_DIR = CATEGORY_DIR / "clustered_graph_forced"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_file = SUBSETS_DIR / f"{CATEGORY}_articles_es_disjoint.csv"
relations_file = SUBSETS_DIR / "RELATIONS_EXTRAITES" / f"relations_extraites_{CATEGORY}_es.json"
clustered_graph_file = CATEGORY_DIR / f"{CATEGORY}_clustered_graph.pkl"
clustered_provenance_file = CATEGORY_DIR / "provenance" / f"{CATEGORY}_clustered_provenance.pkl"

print(f"📂 Catégorie : {CATEGORY}")
print(f"📁 Entrées   : {CATEGORY_DIR}")
print(f"📁 Sortie    : {OUTPUT_DIR}")

📂 Catégorie : gastronomia
📁 Entrées   : /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS_with_relations/mistral_small_2506/gastronomia
📁 Sortie    : /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS_with_relations/mistral_small_2506/gastronomia/clustered_graph_forced


In [3]:
# ============ CHARGEMENT DES DONNÉES ============
df_category = pd.read_csv(input_file)
dict_existing_relations = json.load(open(relations_file, "r", encoding="utf-8"))

with open(clustered_graph_file, "rb") as f:
    clustered_graph = pickle.load(f)

with open(clustered_provenance_file, "rb") as f:
    clustered_provenance = pickle.load(f)

print(f"Articles dans df_category          : {len(df_category)}")
print(f"Articles avec relation hints       : {len(dict_existing_relations)}")
print(f"Relations dans clustered_graph     : {len(clustered_graph.relations)}")
print(f"Edges dans clustered_graph         : {len(clustered_graph.edges)}")
print(f"Entrées de provenance chargées     : {len(clustered_provenance)}")

Articles dans df_category          : 388
Articles avec relation hints       : 388
Relations dans clustered_graph     : 44971
Edges dans clustered_graph         : 6287
Entrées de provenance chargées     : 44971


In [4]:
# ============ CONSTRUCTION DE L'ENSEMBLE GLOBAL DES HINTS ============
def _normalize_predicate(pred: str) -> str:
    """Normalise un prédicat pour comparaison robuste."""
    pred = unicodedata.normalize("NFKC", pred)
    pred = pred.lower().strip()
    pred = " ".join(pred.split())
    return pred

global_relation_hints = set()
for article_id, hints in dict_existing_relations.items():
    if hints:
        global_relation_hints.update(hints)

normalized_hints = {_normalize_predicate(h) for h in global_relation_hints}

print(f"Relation hints bruts    : {len(global_relation_hints)}")
print(f"Relation hints uniques  : {len(normalized_hints)}")
for h in sorted(list(normalized_hints))[:20]:
    print(f"  - {h}")

Relation hints bruts    : 2847
Relation hints uniques  : 2845
  - abarca la práctica ritual de
  - abarca la región de
  - abarca los estados de
  - abarca los países de
  - aborda como tema central
  - acelera la descomposición de
  - acogió a
  - acompaña a
  - acompaña a la celebración de
  - acompaña a la festividad de
  - acompaña al
  - acompaña las celebraciones de
  - acompañan a
  - aconseja lanzar
  - actuaron en
  - actuó como mascota en
  - actúa en la zona
  - acudían al café para consumir
  - adaptaron recetas de
  - adaptó como


In [9]:
# ============ FONCTION DE FORÇAGE DES PRÉDICATS ============
def force_relation_hints_on_clustered_graph(
    graph: Graph,
    relation_hints: set[str],
    provenance: dict,
) -> tuple[Graph, dict]:
    """
    Pour chaque triplet clusterisé, si l'un des prédicats originaux
    (via relation_clusters) correspond à un hint (comparaison normalisée),
    on remplace le prédicat clusterisé par ce prédicat hint.
    Les triplets identiques sont fusionnés.
    """
    normalized_hint_map = {_normalize_predicate(h): h for h in relation_hints}

    new_relations: set[tuple[str, str, str]] = set()
    new_relation_clusters: dict[tuple[str, str, str], set[tuple[str, str, str]]] = defaultdict(set)
    new_provenance: dict[tuple[str, str, str], set] = defaultdict(set)
    n_forced_p = 0

    for s, p, o in graph.relations:
        dedup_triplet = (s, p, o)
        orig_triplets = graph.relation_clusters.get("\t".join(dedup_triplet), [])

        # Recherche d'un prédicat hint parmi les triplets originaux
        forced_p = None
        for orig_triple in orig_triplets:
            _, p_orig, _ = orig_triple
            p_orig_norm = _normalize_predicate(p_orig)
            if p_orig_norm in normalized_hint_map:
                forced_p = p_orig
                n_forced_p +=1
                break

        final_p = forced_p if forced_p is not None else p
        new_triplet = (s, final_p, o)
        new_relations.add(new_triplet)

        for orig_triple in orig_triplets:
            new_relation_clusters[new_triplet].add(tuple(orig_triple))

        arts = provenance.get(dedup_triplet, set())
        new_provenance[new_triplet].update(arts)
    print(f"Nombre de prédicats forcés : {n_forced_p}")

    # Mise à jour des edges avec les nouveaux prédicats
    new_edges = set(graph.edges)
    new_edges.update(p for _, p, _ in new_relations)

    # Mise à jour des edge_clusters : chaque nouveau prédicat devient un singleton
    new_edge_clusters = {
        k: set(v) if isinstance(v, set) else set(v)
        for k, v in (graph.edge_clusters or {}).items()
    }
    for edge in new_edges:
        if edge not in new_edge_clusters:
            new_edge_clusters[edge] = {edge}

    new_graph = Graph(
        entities=graph.entities,
        edges=new_edges,
        relations=new_relations,
        entity_clusters=graph.entity_clusters,
        edge_clusters=new_edge_clusters,
        entity_metadata=graph.entity_metadata,
        relation_clusters={
            "\t".join(k): [list(t) for t in v]
            for k, v in new_relation_clusters.items()
        },
    )

    return new_graph, dict(new_provenance)


In [10]:
# ============ APPLICATION DU FORÇAGE ============
forced_graph, forced_provenance = force_relation_hints_on_clustered_graph(
    clustered_graph,
    global_relation_hints,
    clustered_provenance,
)

print(f"Relations avant forçage : {len(clustered_graph.relations)}")
print(f"Relations après forçage : {len(forced_graph.relations)}")
print(f"Edges avant forçage     : {len(clustered_graph.edges)}")
print(f"Edges après forçage     : {len(forced_graph.edges)}")
print(f"Edge clusters avant     : {len(clustered_graph.edge_clusters or {})}")
print(f"Edge clusters après     : {len(forced_graph.edge_clusters or {})}")

Nombre de prédicats forcés : 22015
Relations avant forçage : 44971
Relations après forçage : 44971
Edges avant forçage     : 6287
Edges après forçage     : 13639
Edge clusters avant     : 7938
Edge clusters après     : 13639


In [11]:
# ============ SAUVEGARDE DU GRAPHE FORCÉ ============
def graph_to_dict(graph: Graph) -> dict:
    return {
        "entities": sorted(list(graph.entities)),
        "edges": sorted(list(graph.edges)),
        "relations": [list(r) for r in sorted(graph.relations)],
        "entity_clusters": {
            k: sorted(list(v)) for k, v in graph.entity_clusters.items()
        } if graph.entity_clusters else {},
        "edge_clusters": {
            k: sorted(list(v)) for k, v in graph.edge_clusters.items()
        } if graph.edge_clusters else {},
        "entity_metadata": {
            k: sorted(list(v)) for k, v in graph.entity_metadata.items()
        } if graph.entity_metadata else {},
        "relation_clusters": graph.relation_clusters,
    }

forced_pkl_path = OUTPUT_DIR / f"{CATEGORY}_clustered_graph.pkl"
forced_json_path = OUTPUT_DIR / f"{CATEGORY}_clustered_graph.json"

with open(forced_pkl_path, "wb") as f:
    pickle.dump(forced_graph, f)

with open(forced_json_path, "w", encoding="utf-8") as f:
    json.dump(graph_to_dict(forced_graph), f, ensure_ascii=False, indent=2)

print(f"✅ Graphe forcé sauvegardé :")
print(f"   - {forced_pkl_path}")
print(f"   - {forced_json_path}")

✅ Graphe forcé sauvegardé :
   - /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS_with_relations/mistral_small_2506/gastronomia/clustered_graph_forced/gastronomia_clustered_graph.pkl
   - /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS_with_relations/mistral_small_2506/gastronomia/clustered_graph_forced/gastronomia_clustered_graph.json


In [12]:
# ============ SAUVEGARDE DE LA PROVENANCE FORCÉE ============
prov_pkl_path = OUTPUT_DIR / f"{CATEGORY}_clustered_provenance.pkl"
prov_json_path = OUTPUT_DIR / f"{CATEGORY}_clustered_provenance.json"

prov_out = {k: sorted(v) for k, v in forced_provenance.items()}
with open(prov_pkl_path, "wb") as f:
    pickle.dump(prov_out, f)

prov_json = [
    {"subject": s, "predicate": p, "object": o, "articles": sorted(arts)}
    for (s, p, o), arts in forced_provenance.items()
]
with open(prov_json_path, "w", encoding="utf-8") as f:
    json.dump(prov_json, f, ensure_ascii=False, indent=2)

print(f"✅ Provenance forcée sauvegardée :")
print(f"   - {prov_pkl_path}")
print(f"   - {prov_json_path}")

✅ Provenance forcée sauvegardée :
   - /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS_with_relations/mistral_small_2506/gastronomia/clustered_graph_forced/gastronomia_clustered_provenance.pkl
   - /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS_with_relations/mistral_small_2506/gastronomia/clustered_graph_forced/gastronomia_clustered_provenance.json


In [13]:
# ============ VÉRIFICATIONS ET STATISTIQUES ============
original_predicates = {p for _, p, _ in clustered_graph.relations}
forced_predicates = {p for _, p, _ in forced_graph.relations}

modified_count = 0
examples = []
for (s, p, o) in clustered_graph.relations:
    if (s, p, o) not in forced_graph.relations:
        modified_count += 1
        for (sf, pf, of) in forced_graph.relations:
            if sf == s and of == o and pf != p:
                examples.append(((s, p, o), (sf, pf, of)))
                break

print(f"Prédicats originaux  : {len(original_predicates)}")
print(f"Prédicats après forcé: {len(forced_predicates)}")
print(f"Triplets modifiés    : {modified_count}")
print(f"\nExemples de modifications :")
for orig, forced in examples[:10]:
    print(f"  {orig[1]}  →  {forced[1]}")

print(f"\nPrédicats préservés qui sont des hints :")
forced_hint_preds = forced_predicates & global_relation_hints
print(f"  Nombre : {len(forced_hint_preds)}")
for pred in sorted(forced_hint_preds)[:20]:
    print(f"  - {pred}")

Prédicats originaux  : 9083
Prédicats après forcé: 9501
Triplets modifiés    : 5663

Exemples de modifications :
  "es una tradición de"  →  es el paradigma de la tradición de
  fue adquirido por  →  fue adquirida por
  "incluyen en el escudo de"  →  incluye en el escudo de
  "se prepara y consume durante"  →  se consume durante
  "incluyen en el escudo de"  →  incluye en el escudo de
  son [adjective] en  →  son apreciadas en
  "son parte de la tradición culinaria de"  →  forma parte de la tradición culinaria de
  "forma parte de"  →  forma parte de la cultura popular de
  "es mencionado"  →  tiene denominación de origen en
  son huevos y larvas de  →  son larvas de

Prédicats préservés qui sont des hints :
  Nombre : 1441
  - abarca la práctica ritual de
  - acompaña a
  - acompaña al
  - acompañan a
  - actuaron en
  - adaptó con la letra
  - adopta la bandeja paisa como símbolo culinario de
  - adoptó el uso de
  - adoptó la receta de
  - adquirió
  - adquirió en
  - adquirió la fó